# Data exploration

Four folders of photos, each typical photobook theme.

There are three sources of information for each photo:

1. The pixels themselves.
2. Scene labels from Amazon Rekognition, stored in a file next to each photo.
3. EXIF metadata inside the photo: when it was taken, where, and on what camera.

Purpose of this notebook is to discover what each of these sources actually tells us, how reliable it is, and what that means for writing captions.

In [ ]:
import contextlib
import io
import json
import random
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycountry
import reverse_geocoder as rg
from PIL import Image, ImageOps

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "data"
FOLDERS = sorted(p.name for p in DATA.iterdir() if p.is_dir())
FOLDERS

## 1. What files are there

Each photo should have a label file next to it with the same name plus `.json`.
First, check that every photo has its label file and there is nothing else in the folders.

In [ ]:
for folder in FOLDERS:
    imgs = {p.name for p in (DATA / folder).glob("*.jpg")}
    jsons = {p.name.removesuffix(".json") for p in (DATA / folder).glob("*.jpg.json")}
    others = [p.name for p in (DATA / folder).iterdir() if not p.name.endswith((".jpg", ".jpg.json"))]
    print(
        f"{folder:25s} {len(imgs):4d} photos  {len(jsons):4d} label files  "
        f"unmatched: {sorted(imgs ^ jsons)}  other files: {others}"
    )

## 2. Label files

The label files are the raw output of the AWS command line tool. They look like
JSON, but the keys are not in quotes, so a normal JSON parser rejects them.

In [ ]:
raw = (DATA / "day-at-beach" / "Image_1.jpg.json").read_text()
print(raw[:400])

In [ ]:
try:
    json.loads(raw)
except json.JSONDecodeError as e:
    print(f"json.loads fails: {e}")